# Notebook showing a fast and efficient way to load Sentinel-2 L1C data into xarray

In [1]:
import glob
import os
import numpy as np

import pandas as pd
import geopandas as gpd
import xarray as xr

import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
from matplotlib.colors import ListedColormap
from GRSdriver import driver_S2_SAFE as S2

import json
import requests
from datetime import datetime
from multiprocessing import Pool
import subprocess

from eoreader.keywords import CLEAN_OPTICAL
import GRSdriver

opj = os.path.join

GRSdriver.__version__

'1.0.7'

### Download L1C images through S3 protocol, please visit https://documentation.dataspace.copernicus.eu/APIs/S3.html

In [2]:

info = pd.read_csv('/DATA/AERONET/info_aeronet_oc_v15_location.csv',index_col=0)#.to_xarray()
info = info[info['number']>0]
info

,start_date,end_date,number,latitude,longitude,s2_tile
site,,,,,,
ARIAKE_TOWER,2018-02-23,2023-03-19,2001,33.103617,130.271950,52SFB
Bahia_Blanca,2020-01-23,2024-09-03,4936,-39.148000,-61.722000,20HPB
Banana_River,2022-06-16,2025-01-25,2006,28.367000,-80.633000,17RNM
Blyth_NOAH,2016-04-21,2017-09-19,33,55.146399,-1.420859,30UWG
Casablanca_Platform,2019-04-02,2025-03-31,18735,40.717000,1.358000,31TCF
Chesapeake_Bay,2021-10-07,2025-03-31,12554,39.124000,-76.359000,18SUJ
COVE_SEAPRISM,2015-11-15,2016-01-09,163,36.900000,-75.710000,18SVF
Galata_Platform,2015-02-02,2023-12-19,15072,43.044624,28.193190,35TNH
Gloria,2015-01-01,2019-08-07,7607,44.599970,29.359670,35TPK


In [3]:
odir = '/data/satellite/Sentinel-2/L1C/'
site='Venise' #Lake_Erie'
info_ = info.loc[site]
info_

start_date    2015-01-01
end_date      2023-03-12
number              6459
latitude         45.3139
longitude        12.5083
s2_tile            32TQR
Name: Venise, dtype: object

In [5]:



date_start=info_.start_date+'T00:00:00.000Z' 
date_end =info_.end_date+'T00:00:00.000Z'
tile=info_.s2_tile

date_start='2017-01-01T00:00:00.000Z' 
date_end ='2020-09-30T00:00:00.000Z'
#tiles=['20PNC']#'10SEJ''31TFJ']#'31TFJ']['55WFU']
#tiles = ['32TNS','32TNR']
cloud_cover_max = 60      


cmd = []

json = requests.get("https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter=Collection/Name eq 'SENTINEL-2' "+
                " and Attributes/OData.CSC.DoubleAttribute/any(att:att/Name eq 'cloudCover' and att/OData.CSC.DoubleAttribute/Value lt {:.2f})".format(cloud_cover_max)+
                " and Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq 'S2MSI1C')"+
                " and contains(Name,'"+tile+"') "+
                # next line is to keep latest baseline only (comment to get baseline < 4)
                " and (contains(Name,'_N04') or contains(Name,'_N05')) "+
                " and ContentDate/Start gt "+date_start+" and ContentDate/Start lt "+date_end+"&$orderby=ContentDate/Start&$top=1000").json()
df = pd.DataFrame.from_dict(json['value'])



for irow,info in df.iterrows():
    #print(info['Name'])
    date=datetime.strptime(info['Name'].split('_')[2],'%Y%m%dT%H%M%S')
    date.year
    odir_ = opj(odir,tile,str(date.year),str(date.month).zfill(2),str(date.day).zfill(2))
    cmd.append('s3cmd -c ~/.s3cfg sync s3:/'+info['S3Path']+' '+odir_)
cmd

['s3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/01/01/S2A_MSIL1C_20170101T100412_N0500_R122_T32TQR_20230918T135340.SAFE /data/satellite/Sentinel-2/L1C/32TQR/2017/01/01',
 's3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/01/04/S2A_MSIL1C_20170104T101402_N0500_R022_T32TQR_20230926T031104.SAFE /data/satellite/Sentinel-2/L1C/32TQR/2017/01/04',
 's3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/01/11/S2A_MSIL1C_20170111T100351_N0500_R122_T32TQR_20230918T063927.SAFE /data/satellite/Sentinel-2/L1C/32TQR/2017/01/11',
 's3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/01/14/S2A_MSIL1C_20170114T101351_N0500_R022_T32TQR_20230918T102847.SAFE /data/satellite/Sentinel-2/L1C/32TQR/2017/01/14',
 's3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/01/24/S2A_MSIL1C_20170124T101311_N0500_R022_T32TQR_20231026T031702.SAFE /data/satellite/Sentinel-2/L1C/32TQR/2017/01/24',
 's3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C

In [6]:

def call(command):
    print(command)
    pipeline_out = subprocess.call(command, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT, shell=True)
    return

ncore=4
with Pool(processes=ncore) as pool:
    pool.map(call, cmd)
    pool.close

s3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/01/01/S2A_MSIL1C_20170101T100412_N0500_R122_T32TQR_20230918T135340.SAFE /data/satellite/Sentinel-2/L1C/32TQR/2017/01/01s3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/09/08/S2A_MSIL1C_20170908T100031_N0500_R122_T32TQR_20230929T065240.SAFE /data/satellite/Sentinel-2/L1C/32TQR/2017/09/08s3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/04/29/S2B_MSIL1C_20170429T101019_N0500_R022_T32TQR_20231114T145340.SAFE /data/satellite/Sentinel-2/L1C/32TQR/2017/04/29s3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/07/10/S2A_MSIL1C_20170710T100031_N0500_R122_T32TQR_20231004T212359.SAFE /data/satellite/Sentinel-2/L1C/32TQR/2017/07/10



s3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/01/04/S2A_MSIL1C_20170104T101402_N0500_R022_T32TQR_20230926T031104.SAFE /data/satellite/Sentinel-2/L1C/32TQR/2017/01/04
s3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/09/13/S2B_

In [16]:
import requests

url = f"https://zipper.dataspace.copernicus.eu/odata/v1/Products("+df.Id[0]+")/$value"

headers = {"Authorization": f"Bearer {access_token}"}

session = requests.Session()
session.headers.update(headers)
response = session.get(url, headers=headers, stream=True)

with open("product.zip", "wb") as file:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            file.write(chunk)